# 1. Titulo y proposito

Este notebook documenta de forma academica y reproducible el entrenamiento y la evaluacion del modelo supervisado de condicion de Gradix a partir del dataset tabular generado por el pipeline tecnico del proyecto.

La tarea supervisada consiste en predecir `target_damaged`, una variable operacional binaria derivada de la organizacion del dataset (`1 = damaged`, `0 = undamaged`). El objetivo no es reemplazar un proceso profesional de grading, sino estudiar si las features extraidas por Gradix permiten distinguir ambas clases de manera consistente.

# 2. Modelacion supervisada en Gradix

En este proyecto, **modelacion supervisada** significa entrenar un algoritmo con ejemplos etiquetados para aprender una relacion entre variables de entrada y una salida objetivo.

- Las entradas son features tabulares derivadas del pipeline de vision por computadora de Gradix.
- El target es `target_damaged`.
- El notebook entrena y evalua modelos.
- La app (`app.py`) no entrena: solo reutiliza artefactos ya guardados para inferencia.

Esta separacion mantiene el entrenamiento como un proceso reproducible, auditable y offline, mientras la app queda enfocada en uso interactivo e inferencia.

# 3. Configuracion del entorno

La siguiente celda localiza la raiz del repo, define rutas, crea `artifacts/` y prepara imports. Si `scikit-learn` no esta instalado, el notebook lanza un error explicito para que el entorno pueda completarse de forma controlada.

In [ ]:
# Opcional: instala dependencias si hacen falta en tu entorno local.
# !pip install pandas numpy scikit-learn joblib matplotlib

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

try:
    import joblib
    from sklearn.base import clone
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        ConfusionMatrixDisplay,
        accuracy_score,
        balanced_accuracy_score,
        confusion_matrix,
        f1_score,
        precision_score,
        recall_score,
    )
    from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'Este notebook requiere scikit-learn y joblib. Instala dependencias con: pip install scikit-learn joblib'
    ) from exc

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 130

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'data' / 'processed' / 'dataset_gradix.csv').exists() and (candidate / 'app.py').exists():
            return candidate
    raise FileNotFoundError('No se encontro la raiz del repo Gradix desde el directorio actual.')

ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATASET_PATH = ROOT / 'data' / 'processed' / 'dataset_gradix.csv'
ARTIFACTS_DIR = ROOT / 'artifacts'
MODEL_PATH = ARTIFACTS_DIR / 'condition_model.joblib'
FEATURES_PATH = ARTIFACTS_DIR / 'feature_names.json'
METRICS_PATH = ARTIFACTS_DIR / 'metrics.json'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_FOLDS = 5

print('ROOT =', ROOT)
print('DATASET_PATH =', DATASET_PATH)
print('ARTIFACTS_DIR =', ARTIFACTS_DIR)

In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'No se encontro el dataset esperado: {DATASET_PATH}')

df = pd.read_csv(DATASET_PATH)
print('Shape original:', df.shape)
display(df.head())

# 4. Limitaciones del dataset

Antes de modelar, conviene dejar explicitadas varias limitaciones:

- El target `target_damaged` es una etiqueta operacional derivada de la estructura de carpetas del dataset, no una certificacion profesional de grading.
- Puede haber multiples fotos de la misma carta, por lo que la separacion train/test es reproducible pero no necesariamente independiente a nivel de instancia fisica.
- El pipeline incluye filtros tecnicos y heuristicas de calidad; por eso el filtrado previo al entrenamiento debe ser documentado con cuidado.
- El dataset puede reflejar sesgos de fondo, iluminacion, distancia, blur y condiciones de captura.

# 5. Filtrado explicito de filas validas

Este notebook aplica una politica de filtrado en dos niveles:

1. **Filtro estricto preferido**: usar `usable_for_condition_model == True` y `target_damaged` disponible. Este es el criterio tecnico mas directo cuando el pipeline ya marca muestras aptas para el modelo.
2. **Fallback documentado**: si el filtro estricto no deja suficientes muestras para entrenar, se usa un subconjunto tecnico alternativo con `postwarp_valid == True`, `features_computed == True`, `warp_success == True` y `target_damaged` disponible.

De esta manera queda claro **cuando** se usa `usable_for_condition_model`: se intenta primero como filtro principal, pero solo si realmente produce una muestra entrenable con ambas clases presentes.

In [ ]:
strict_mask = (
    df['target_damaged'].notna()
    & df['usable_for_condition_model'].fillna(False).astype(bool)
)

fallback_mask = (
    df['target_damaged'].notna()
    & df['warp_success'].fillna(False).astype(bool)
    & df['features_computed'].fillna(False).astype(bool)
    & df['postwarp_valid'].fillna(False).astype(bool)
)

strict_df = df.loc[strict_mask].copy()
fallback_df = df.loc[fallback_mask].copy()

def class_counts(frame: pd.DataFrame) -> dict:
    if frame.empty:
        return {}
    return frame['target_damaged'].astype(int).value_counts(dropna=False).to_dict()

strict_counts = class_counts(strict_df)
fallback_counts = class_counts(fallback_df)

def enough_for_training(counts: dict, min_per_class: int = 20) -> bool:
    return len(counts) == 2 and min(counts.values()) >= min_per_class

if enough_for_training(strict_counts):
    model_df = strict_df.copy()
    filtering_mode = 'strict_usable_for_condition_model'
    filtering_description = 'Se uso usable_for_condition_model como filtro primario.'
elif enough_for_training(fallback_counts):
    model_df = fallback_df.copy()
    filtering_mode = 'fallback_postwarp_valid'
    filtering_description = (
        'No hubo suficientes filas con usable_for_condition_model=True; '
        'se uso fallback tecnico con warp_success, features_computed y postwarp_valid.'
    )
else:
    raise ValueError(
        f'No hay suficientes muestras entrenables. strict={strict_counts}, fallback={fallback_counts}'
    )

filter_report = pd.Series({
    'filas_dataset_original': int(len(df)),
    'filas_filtro_estricto': int(len(strict_df)),
    'filas_filtro_fallback': int(len(fallback_df)),
    'filtering_mode': filtering_mode,
    'filtering_description': filtering_description,
    'strict_class_counts': strict_counts,
    'fallback_class_counts': fallback_counts,
    'selected_shape': tuple(model_df.shape),
}, name='valor')
display(filter_report.to_frame())

# 6. Exclusiones de leakage y seleccion de features

Para evitar leakage, el modelo excluye columnas que identifican directamente la clase, resumen decisiones finales del pipeline o contienen rutas/nombres de archivo.

Se excluyen de forma explicita:

- `target_damaged`
- `label_condition`
- `analysis_status`
- `invalid_reasons`
- rutas y nombres de archivo
- banderas de exito del pipeline y columnas derivadas de decision final
- familias `score_` y `capture_assessment_`, ya que son agregados heuristicas de alto nivel y no features base crudas

Despues de excluirlas, el notebook se queda con columnas numericas y booleanas convertibles a valores numericos.

In [ ]:
EXCLUDE_EXACT = {
    'target_damaged',
    'label_condition',
    'analysis_status',
    'invalid_reasons',
    'usable_for_condition_model',
    'image_id',
    'image_filename',
    'image_path',
    'relative_path_from_raw',
    'categoria_carpeta_raw',
    'feature_schema_version',
    'run_timestamp',
    'pipeline_stage',
    'error',
    'procesado_exito',
    'det_success',
    'warp_success',
    'postwarp_computed',
    'features_computed',
    'scores_computed',
    'postwarp_valid',
    'postwarp_score',
    'postwarp_retry_recommended',
}

EXCLUDE_PREFIXES = (
    'score_',
    'capture_assessment_',
)

numeric_candidates = model_df.select_dtypes(include=['number', 'bool']).columns.tolist()
feature_cols = [
    column_name
    for column_name in numeric_candidates
    if column_name not in EXCLUDE_EXACT and not any(column_name.startswith(prefix) for prefix in EXCLUDE_PREFIXES)
]

X = model_df[feature_cols].copy().astype(float)
y = model_df['target_damaged'].astype(int).copy()

feature_summary = pd.Series({
    'n_numeric_candidates': int(len(numeric_candidates)),
    'n_excluded_exact': int(len([c for c in numeric_candidates if c in EXCLUDE_EXACT])),
    'n_selected_features': int(len(feature_cols)),
}, name='valor')
display(feature_summary.to_frame())
print('Primeras features seleccionadas:')
print(feature_cols[:60])

# 7. Separacion train/test reproducible

El experimento usa `train_test_split` estratificado con `random_state=42`. La estratificacion mantiene la proporcion de clases entre entrenamiento y prueba, lo cual es especialmente importante cuando el dataset filtrado puede quedar desbalanceado.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_report = pd.DataFrame([
    {'subset': 'train', 'rows': int(len(X_train)), 'class_counts': y_train.value_counts().to_dict()},
    {'subset': 'test', 'rows': int(len(X_test)), 'class_counts': y_test.value_counts().to_dict()},
    ])
display(split_report)

# 8. Modelos base comparados

Se comparan tres modelos base comunmente usados en clasificacion tabular:

- **Logistic Regression** como baseline lineal interpretable.
- **Random Forest** como baseline no lineal basado en ensambles de arboles.
- **HistGradientBoostingClassifier** como baseline boosting para relaciones no lineales.

La seleccion del mejor modelo se basa en validacion cruzada sobre el conjunto de entrenamiento, priorizando `balanced_accuracy` y `f1`. Luego se reportan metricas finales sobre test.

In [ ]:
numeric_transformer_scaled = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ])

numeric_transformer_unscaled = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ])

pre_scaled = ColumnTransformer([('num', numeric_transformer_scaled, feature_cols)], remainder='drop')
pre_unscaled = ColumnTransformer([('num', numeric_transformer_unscaled, feature_cols)], remainder='drop')

models = {
    'logistic_regression': Pipeline([
        ('prep', pre_scaled),
        ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)),
    ]),
    'random_forest': Pipeline([
        ('prep', pre_unscaled),
        ('model', RandomForestClassifier(
            n_estimators=400,
            min_samples_leaf=2,
            class_weight='balanced_subsample',
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),
    'hist_gradient_boosting': Pipeline([
        ('prep', pre_unscaled),
        ('model', HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_depth=6,
            max_iter=300,
            random_state=RANDOM_STATE,
        )),
    ]),
}

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    'accuracy': 'accuracy',
    'balanced_accuracy': 'balanced_accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
}

cv_rows = []
for model_name, estimator in models.items():
    scores = cross_validate(estimator, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_rows.append({
        'model': model_name,
        'cv_accuracy_mean': float(np.mean(scores['test_accuracy'])),
        'cv_balanced_accuracy_mean': float(np.mean(scores['test_balanced_accuracy'])),
        'cv_precision_mean': float(np.mean(scores['test_precision'])),
        'cv_recall_mean': float(np.mean(scores['test_recall'])),
        'cv_f1_mean': float(np.mean(scores['test_f1'])),
    })

cv_results = pd.DataFrame(cv_rows).sort_values(
    ['cv_balanced_accuracy_mean', 'cv_f1_mean', 'cv_accuracy_mean'],
    ascending=False,
    ).reset_index(drop=True)
display(cv_results)

# 9. Entrenamiento final y evaluacion en test

Cada modelo se ajusta sobre el conjunto de entrenamiento y se evalua sobre el conjunto de prueba. Las metricas reportadas son:

- accuracy
- balanced accuracy
- precision
- recall
- f1
- matriz de confusion

El modelo final se elige a partir del ranking de validacion cruzada y luego se guarda en `artifacts/`.

In [ ]:
test_rows = []
trained_models = {}
predictions_by_model = {}

for model_name, estimator in models.items():
    fitted = clone(estimator)
    fitted.fit(X_train, y_train)
    y_pred = fitted.predict(X_test)

    trained_models[model_name] = fitted
    predictions_by_model[model_name] = y_pred

    test_rows.append({
        'model': model_name,
        'accuracy': float(accuracy_score(y_test, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_test, y_pred)),
        'precision': float(precision_score(y_test, y_pred, zero_division=0)),
        'recall': float(recall_score(y_test, y_pred, zero_division=0)),
        'f1': float(f1_score(y_test, y_pred, zero_division=0)),
    })

test_results = pd.DataFrame(test_rows).sort_values(
    ['balanced_accuracy', 'f1', 'accuracy'],
    ascending=False,
    ).reset_index(drop=True)
display(test_results)

best_model_name = cv_results.iloc[0]['model']
best_model = trained_models[best_model_name]
y_pred_best = predictions_by_model[best_model_name]

print('Modelo seleccionado por CV:', best_model_name)

In [ ]:
best_metrics = {
    'accuracy': float(accuracy_score(y_test, y_pred_best)),
    'balanced_accuracy': float(balanced_accuracy_score(y_test, y_pred_best)),
    'precision': float(precision_score(y_test, y_pred_best, zero_division=0)),
    'recall': float(recall_score(y_test, y_pred_best, zero_division=0)),
    'f1': float(f1_score(y_test, y_pred_best, zero_division=0)),
}

pd.Series(best_metrics, name=best_model_name)

In [ ]:
cm = confusion_matrix(y_test, y_pred_best)
cm_df = pd.DataFrame(cm, index=['true_0', 'true_1'], columns=['pred_0', 'pred_1'])
display(cm_df)

fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['undamaged', 'damaged']).plot(ax=ax, values_format='d')
ax.set_title(f'Matriz de confusion - {best_model_name}')
plt.show()

# 10. Guardado de artefactos

El notebook guarda tres artefactos principales:

- `artifacts/condition_model.joblib`: modelo entrenado final.
- `artifacts/feature_names.json`: lista ordenada de features usadas en entrenamiento.
- `artifacts/metrics.json`: resumen reproducible del experimento y sus metricas.

La app no entrena modelos porque el entrenamiento requiere control experimental, versionado de dataset, exclusion de leakage y evaluacion reproducible. La app queda reservada para inferencia con un artefacto ya consolidado.

In [ ]:
joblib.dump(best_model, MODEL_PATH)
FEATURES_PATH.write_text(json.dumps(feature_cols, indent=2, ensure_ascii=False), encoding='utf-8')

metrics_payload = {
    'dataset_path': str(DATASET_PATH),
    'filtering_mode': filtering_mode,
    'filtering_description': filtering_description,
    'rows_original': int(len(df)),
    'rows_modeling': int(len(model_df)),
    'test_size': TEST_SIZE,
    'random_state': RANDOM_STATE,
    'cv_folds': CV_FOLDS,
    'n_features': int(len(feature_cols)),
    'excluded_exact_columns': sorted(EXCLUDE_EXACT),
    'excluded_prefixes': list(EXCLUDE_PREFIXES),
    'train_class_counts': {str(k): int(v) for k, v in y_train.value_counts().to_dict().items()},
    'test_class_counts': {str(k): int(v) for k, v in y_test.value_counts().to_dict().items()},
    'cv_results': cv_results.to_dict(orient='records'),
    'test_results': test_results.to_dict(orient='records'),
    'best_model_name': best_model_name,
    'best_model_metrics': best_metrics,
}

METRICS_PATH.write_text(json.dumps(metrics_payload, indent=2, ensure_ascii=False), encoding='utf-8')

print('Modelo guardado en:', MODEL_PATH)
print('Features guardadas en:', FEATURES_PATH)
print('Metricas guardadas en:', METRICS_PATH)

# 11. Cierre metodologico

Este notebook deja documentado un flujo reproducible de modelado supervisado para Gradix: carga de dataset, filtrado explicito, exclusion de leakage, separacion train/test, comparacion de modelos base, evaluacion final y guardado de artefactos.

Su rol es experimental y academico. Por diseno, la app no entrena; la app solo infiere sobre nuevas capturas usando un modelo previamente entrenado y guardado.

In [ ]:
artifact_report = pd.Series({
    'condition_model.joblib exists': MODEL_PATH.exists(),
    'feature_names.json exists': FEATURES_PATH.exists(),
    'metrics.json exists': METRICS_PATH.exists(),
    'selected_model': best_model_name,
    'filtering_mode': filtering_mode,
    'n_features': int(len(feature_cols)),
}, name='valor')
display(artifact_report.to_frame())